# Per-rank read statistics: 10 TiB, 640 ranks (job 8657651)

Validates and summarizes the per-rank fio JSONs from the MPI read benchmark
`read_fio-10tib-fs16g_n20_ppn32_bs2m_iod16_8657651` (20 nodes x 32 ranks,
one rank per 16 GiB file, bs=2m, iodepth=16, one iteration).

**Validation** — every rank file must parse, report `error=0`, have read exactly
16 GiB with no short/dropped IOs, and ranks 0..639 must all be present.

**Statistics** — aggregate bandwidth two ways plus per-rank distributions:
- **Wall-clock aggregate**: total bytes / (last rank end - first rank start), using
  each job's `job_start` + `runtime`. The honest number, bounded by stragglers.
- **Sum of per-rank `bw_bytes`**: optimistic upper bound; each rank's average only
  applies while that rank was running, so skew is ignored.

fio units: `read.bw_bytes` is B/s (`bw` is the same in KiB/s), `read.runtime` is ms,
`lat_ns.mean` is the mean completion latency in ns.

In [1]:
import json
import re
import statistics as st
from pathlib import Path

RUN_NAME = "read_fio-10tib-fs16g_n20_ppn32_bs2m_iod16_8657651"
EXPECT_RANKS = 640
EXPECT_BYTES = 16 * 1024**3   # SIZE=16g per rank
GiB = 1024**3

# Run folder sits next to this notebook.
RESULT_DIR = Path(RUN_NAME)
files = sorted(RESULT_DIR.glob("*.json"))
print(f"{len(files)} rank files in {RESULT_DIR}")

640 rank files in read_fio-10tib-fs16g_n20_ppn32_bs2m_iod16_8657651


## Validation: did every rank read its full 16 GiB?

In [2]:
problems = []
rows = []   # one dict per rank

for fp in files:
    txt = fp.read_text()
    try:
        d = json.loads(txt[txt.index("{"):])   # skip any non-JSON header line
    except (ValueError, json.JSONDecodeError) as e:
        problems.append(f"{fp.name}: unparseable JSON ({e})")
        continue
    rank = int(re.search(r"rank(\d+)", fp.name).group(1))
    if len(d["jobs"]) != 1:
        problems.append(f"{fp.name}: {len(d['jobs'])} jobs, expected 1")
    j = d["jobs"][0]
    r = j["read"]
    if j["error"] != 0:
        problems.append(f"{fp.name}: fio error={j['error']}")
    if r["io_bytes"] != EXPECT_BYTES:
        problems.append(f"{fp.name}: read {r['io_bytes']} B, expected {EXPECT_BYTES}")
    if r["short_ios"] or r["drop_ios"]:
        problems.append(f"{fp.name}: short_ios={r['short_ios']} drop_ios={r['drop_ios']}")
    if j["jobname"] != f"large.{rank}":
        problems.append(f"{fp.name}: jobname {j['jobname']} != large.{rank}")
    rows.append({
        "rank": rank,
        "io_bytes": r["io_bytes"],
        "runtime_ms": r["runtime"],
        "bw_bytes": r["bw_bytes"],
        "lat_mean_ns": r["lat_ns"]["mean"],
        "start_ms": j.get("job_start", d["timestamp_ms"]),
    })

ranks_seen = {r["rank"] for r in rows}
missing = set(range(EXPECT_RANKS)) - ranks_seen
if missing:
    problems.append(f"missing ranks: {sorted(missing)}")

if problems:
    print(f"PROBLEMS ({len(problems)}):")
    for p in problems:
        print(" -", p)
else:
    print(f"All reads successful: {len(rows)} ranks, error=0, "
          f"io_bytes={EXPECT_BYTES/GiB:.0f} GiB each, no short/dropped IOs, "
          f"ranks 0-{EXPECT_RANKS-1} complete.")

All reads successful: 640 ranks, error=0, io_bytes=16 GiB each, no short/dropped IOs, ranks 0-639 complete.


## Aggregate bandwidth

In [3]:
total_bytes = sum(r["io_bytes"] for r in rows)
starts = [r["start_ms"] for r in rows]
ends = [r["start_ms"] + r["runtime_ms"] for r in rows]
wall_s = (max(ends) - min(starts)) / 1000
sum_bw = sum(r["bw_bytes"] for r in rows)

print(f"Total read              : {total_bytes/1024**4:.2f} TiB in {len(rows)} ranks")
print(f"Wall clock              : {wall_s:.1f} s (first rank start -> last rank end)")
print(f"Aggregate bandwidth     : {total_bytes/wall_s/GiB:.1f} GiB/s = {total_bytes/wall_s/1e9:.1f} GB/s")
print(f"Sum of per-rank bw      : {sum_bw/GiB:.1f} GiB/s (upper bound, ignores skew)")

Total read              : 10.00 TiB in 640 ranks
Wall clock              : 107.9 s (first rank start -> last rank end)
Aggregate bandwidth     : 94.9 GiB/s = 101.9 GB/s
Sum of per-rank bw      : 1108.8 GiB/s (upper bound, ignores skew)


## Per-rank distributions

In [4]:
runtimes_s = [r["runtime_ms"] / 1000 for r in rows]
bws_gib = [r["bw_bytes"] / GiB for r in rows]
lats_ms = [r["lat_mean_ns"] / 1e6 for r in rows]

def pctl(vals, p):
    s = sorted(vals)
    return s[min(len(s) - 1, int(round(p / 100 * (len(s) - 1))))]

hdr = f"{'metric':<22} {'min':>8} {'p50':>8} {'mean':>8} {'p99':>8} {'max':>8} {'stdev':>8}"
print(hdr)
print("-" * len(hdr))
for name, vals in [("runtime (s)", runtimes_s),
                   ("bandwidth (GiB/s)", bws_gib),
                   ("mean latency (ms)", lats_ms)]:
    print(f"{name:<22} {min(vals):>8.2f} {st.median(vals):>8.2f} {st.mean(vals):>8.2f} "
          f"{pctl(vals, 99):>8.2f} {max(vals):>8.2f} {st.stdev(vals):>8.2f}")

metric                      min      p50     mean      p99      max    stdev
----------------------------------------------------------------------------
runtime (s)                2.37     9.87    11.51    34.94    41.08     6.74
bandwidth (GiB/s)          0.39     1.62     1.73     3.62     6.74     0.72
mean latency (ms)          4.61    19.24    22.44    68.19    80.19    13.16


## Stragglers

The slowest ranks bound the wall-clock aggregate: the median rank finishes in
~10 s but the tail stretches the pass to ~108 s, which is why the wall-clock
aggregate sits far below the summed per-rank bandwidths.

In [5]:
slowest = sorted(rows, key=lambda r: -r["runtime_ms"])[:10]
print(f"{'rank':>5} {'node':>5} {'runtime s':>10} {'bw GiB/s':>9} {'mean lat ms':>12}")
for r in slowest:
    print(f"{r['rank']:>5} {r['rank']//32:>5} {r['runtime_ms']/1000:>10.1f} "
          f"{r['bw_bytes']/GiB:>9.2f} {r['lat_mean_ns']/1e6:>12.1f}")

 rank  node  runtime s  bw GiB/s  mean lat ms
  608    19       41.1      0.39         80.2
  462    14       37.7      0.42         73.5
  402    12       37.2      0.43         72.6
  494    15       36.8      0.43         71.8
  618    19       36.3      0.44         70.9
   59     1       35.8      0.45         69.9
  461    14       34.9      0.46         68.2
  115     3       34.8      0.46         67.9
  201     6       34.6      0.46         67.6
  171     5       34.4      0.47         67.1
